# Milvus 数据存储与检索 - 关键API知识点

## 一、Schema 设计（milvus_db_with_schema.py）


### 1.1 字段定义

```python
schema = client.create_schema()

# 基础字段
schema.add_field("id", DataType.INT64, is_primary=True, auto_id=True)
schema.add_field("category", DataType.VARCHAR, max_length=1000)  # text/image
schema.add_field("filename", DataType.VARCHAR, max_length=1000)
schema.add_field("filetype", DataType.VARCHAR, max_length=1000)  # pdf/md

# 文本字段 - 启用分词器
schema.add_field("title", DataType.VARCHAR, max_length=1000, 
                enable_analyzer=True,
                analyzer_params={'tokenizer': 'jieba', 'filter': ['cnalphanumonly']})
schema.add_field("text", DataType.VARCHAR, max_length=10000,
                enable_analyzer=True,
                analyzer_params={'tokenizer': 'jieba', 'filter': ['cnalphanumonly']})

# 向量字段
schema.add_field("title_sparse", DataType.SPARSE_FLOAT_VECTOR)  # BM25稀疏向量
schema.add_field("text_content_sparse", DataType.SPARSE_FLOAT_VECTOR)
schema.add_field("text_content_dense", DataType.FLOAT_VECTOR, dim=1024)  # DashScope密集向量
```


### 1.2 BM25 Function

```python
# BM25自动将文本转为稀疏向量
title_bm25_function = Function(
    name="title_bm25_emb",
    input_field_names=["title"],
    output_field_names=["title_sparse"],
    function_type=FunctionType.BM25
)
schema.add_function(title_bm25_function)

content_bm25_function = Function(
    name="text_content_bm25_emb",
    input_field_names=["text"],
    output_field_names=["text_content_sparse"],
    function_type=FunctionType.BM25
)
schema.add_function(content_bm25_function)
```

**关键点**：只需插入文本，Milvus自动生成稀疏向量


### 1.3 索引配置

```python
index_params = client.prepare_index_params()

# 稀疏向量索引 - BM25
index_params.add_index(
    field_name="text_content_sparse",
    index_type="SPARSE_INVERTED_INDEX",
    metric_type="BM25",
    params={
        "inverted_index_algo": "DAAT_MAXSCORE",
        "bm25_k1": 1.2,      # 词频饱和度
        "bm25_b": 0.75       # 文档长度归一化
    }
)

# 稠密向量索引 - HNSW
index_params.add_index(
    field_name="text_content_dense",
    index_type="HNSW",
    metric_type="COSINE",
    params={
        "M": 16,              # 每节点最大连接数
        "efConstruction": 200 # 构建时搜索候选数
    }
)
```


### 1.4 数据插入流程

```python
# 数据格式
data = [
    {
        'text': '标题:内容',           # 文本内容（图片则为LLM生成的描述）
        'title': 'Header1 --> Header2',
        'category': 'text',           # text/image
        'filename': 'xxx.pdf',
        'filetype': 'pdf',
        'image_path': '',             # 图片路径
        'text_content_dense': [...]   # 1024维向量（DashScope API生成）
    }
]

# 插入（BM25稀疏向量自动生成）
client.insert(collection_name=COLLECTION_NAME, data=data)
```

**关键点**：
- `text_content_dense` 需要手动通过DashScope API生成
- `text_content_sparse` 和 `title_sparse` 由BM25 Function自动生成


## 二、检索方法（milvus_retrieve.py）


### 2.1 密集向量检索（Dense Search）

```python
def dense_search(self, query_embedding, limit=5):
    search_params = {"metric_type": "COSINE", "params": {"nprobe": 10}}
    res = self.client.search(
        collection_name=self.collection_name,
        data=[query_embedding],        # 查询向量
        anns_field="text_content_dense",
        limit=limit,
        search_params=search_params,
        output_fields=["text", "category", "filename", "image_path", "title"],
    )
    return res[0]
```

**适用场景**：图片查询（图片转向量后检索）


### 2.2 稀疏向量检索（Sparse Search）

```python
def sparse_content_search(self, query, limit=5):
    search_params = {"metric_type": "BM25", "params": {'drop_ratio_search': 0.2}}
    res = self.client.search(
        collection_name=self.collection_name,
        data=[query],                  # 原始文本
        anns_field="text_content_sparse",
        limit=limit,
        search_params=search_params,
        output_fields=["text", "category", "filename", "image_path", "title"],
    )
    return res[0]
```

**关键点**：直接传入文本，Milvus自动用BM25转为稀疏向量检索


### 2.3 混合检索（Hybrid Search）⭐

```python
def hybrid_search(self, query_dense_embedding, query_text, 
                 sparse_weight=1.0, dense_weight=1.0, limit=10):
    # 密集向量请求
    dense_req = AnnSearchRequest(
        data=[query_dense_embedding],
        anns_field="text_content_dense",
        limit=limit,
        param={"metric_type": "COSINE", "params": {"nprobe": 10}}
    )
    
    # 稀疏向量请求
    sparse_req = AnnSearchRequest(
        data=[query_text],
        anns_field="text_content_sparse",
        limit=limit,
        param={"metric_type": "BM25", "params": {'drop_ratio_search': 0.2}}
    )
    
    # 加权重排
    ranker = WeightedRanker(sparse_weight, dense_weight)
    
    res = self.client.hybrid_search(
        collection_name=self.collection_name,
        reqs=[dense_req, sparse_req],
        ranker=ranker,
        limit=limit,
        output_fields=["text", "category", "filename", "image_path", "title"],
    )[0]
    return res
```

**关键点**：
- 同时利用语义相似度（dense）和关键词匹配（sparse）
- `WeightedRanker` 融合两种检索结果
- 也可用 `RRFRanker(k=100)` 进行倒数排名融合


### 2.4 统一检索接口

```python
def retrieve(self, query: str) -> List[Dict[str, Any]]:
    # 判断是图片还是文本
    if os.path.isfile(query):
        # 图片 -> base64 -> DashScope API -> dense向量
        base64_img, _ = image_to_base64(query)
        input_data = [{'image': base64_img}]
        ok, dense_embedding, status, retry_after = call_dashscope_once(input_data)
        results = self.dense_search(dense_embedding, limit=self.top_k)
    else:
        # 文本 -> DashScope API -> dense向量
        input_data = [{'text': query}]
        ok, dense_embedding, status, retry_after = call_dashscope_once(input_data)
        results = self.hybrid_search(dense_embedding, query, limit=self.top_k)
    
    # 格式化返回
    docs = []
    for hit in results:
        docs.append({
            "text": hit.text,
            "category": hit.category,
            "filename": hit.filename,
            "image_path": hit.image_path,
            "title": hit.title
        })
    return docs
```

**检索策略**：
- 图片查询 → 纯dense检索
- 文本查询 → 混合检索（dense + sparse）


## 三、核心流程总结


### 3.1 数据入库流程

```
Document对象
  ↓
doc_to_dict() - 转为字典格式
  ↓
generate_image_description() - 图片用LLM生成文本描述
  ↓
process_item_with_guard() - 调用DashScope API生成dense向量
  ↓
write_to_milvus() - 插入数据库（BM25自动生成sparse向量）
```


### 3.2 检索流程

```
用户查询（文本/图片路径）
  ↓
判断类型
  ↓
├─ 图片 → image_to_base64() → DashScope API → dense向量 → dense_search()
  ↓
└─ 文本 → DashScope API → dense向量 → hybrid_search(dense向量 + 原始文本)
  ↓
返回结果列表
```


## 四、关键参数说明


### 4.1 BM25参数

- `bm25_k1=1.2`：词频饱和度，越大表示词频影响越大
- `bm25_b=0.75`：文档长度归一化，0表示不归一化，1表示完全归一化
- `drop_ratio_search=0.2`：检索时丢弃低分结果比例


### 4.2 HNSW参数

- `M=16`：每个节点最大连接数，越大精度越高但内存占用越大
- `efConstruction=200`：构建索引时的搜索候选数，越大构建越慢但质量越高
- `nprobe=10`：搜索时探测的聚类数


### 4.3 向量维度

- DashScope多模态向量：1024维
- BM25稀疏向量：动态维度（词汇表大小）


## 五、使用示例


In [ ]:
from pymilvus import MilvusClient
from milvus_db.milvus_retrieve import MilvusRetriever
from env_utils import COLLECTION_NAME, MILVUS_URI

# 初始化
client = MilvusClient(uri=MILVUS_URI, user='root', password='Milvus')
retriever = MilvusRetriever(
    collection_name=COLLECTION_NAME,
    milvus_client=client,
    top_k=5
)

# 文本检索
docs = retriever.retrieve("GPT-4's accuracy scores")

# 图片检索
docs = retriever.retrieve("path/to/image.jpg")


## 六、重要注意事项


1. **向量生成**：
   - Dense向量：需手动调用DashScope API
   - Sparse向量：Milvus BM25 Function自动生成

2. **检索策略**：
   - 图片查询用dense_search（因为没有文本做BM25）
   - 文本查询用hybrid_search（结合语义和关键词）

3. **数据长度限制**：
   - text字段最大10000字符
   - title字段最大1000字符

4. **图片处理**：
   - 图片需先用LLM生成文本描述
   - 描述文本存入text字段用于向量化
   - 原始图片路径存入image_path字段
